In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from pipeline_helpers import ARTIFACT_DIR, DATA_PATH, SEED

print(f"Data: {DATA_PATH}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Reproducibility seed: {SEED}")
from pipeline_helpers import phase3
result = phase3()
print(f"Training-only feature selection rows: {result['manifest']['feature_selection_rows']}")
print(f"Training-only feature selection rows: {result['manifest']['feature_selection_rows']}")
print("Sample standard deviation for each climate variable:")
display(result["std_table"])

Data: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\data\raw\europe_cereal_yield_climate_1990_2022.csv
Artifacts: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline
Reproducibility seed: 42
Training-only feature selection rows: 2596
Training-only feature selection rows: 2596
Sample standard deviation for each climate variable:


,feature,std
0,WB_CCKP_HD40,0.000000
1,WB_CCKP_HI39,0.000000
2,WB_CCKP_HD50,0.000000
3,WB_CCKP_HD42,0.000000
4,WB_CCKP_HI41,0.000000
5,WB_CCKP_HI37,0.000278
6,WB_CCKP_HD45,0.002908
7,WB_CCKP_HD35,0.009092
8,WB_CCKP_HI35,0.010181
9,WB_CCKP_TX84RR,0.092649


## Near-zero variance removal

Climate columns with sample standard deviation below 0.02 are excluded before clustering. Variance, clustering, target correlation, and VIF are calculated on the training partition only; the fixed stratified split is recreated and saved in Phase 4.

In [2]:
print(f"Near-zero-variance features dropped: {len(result['low_variance'])}")
display(result["low_variance"])

Near-zero-variance features dropped: 9


,feature,std
0,WB_CCKP_HD40,0.000000
1,WB_CCKP_HI39,0.000000
2,WB_CCKP_HD50,0.000000
3,WB_CCKP_HD42,0.000000
4,WB_CCKP_HI41,0.000000
5,WB_CCKP_HI37,0.000278
6,WB_CCKP_HD45,0.002908
7,WB_CCKP_HD35,0.009092
8,WB_CCKP_HI35,0.010181


## Average-linkage clusters

Distances are `1 - |Pearson r|`. Average linkage is cut at distance 0.15, corresponding to the requested 0.85 absolute-correlation scale. Each cluster keeps the member with the strongest absolute marginal correlation with YIELD, measured on training rows only.

In [3]:
display(result["cluster_report"])
display(result["cluster_members"])
display(result["selected_climate"])
display(result["selected_count"])

,cluster,members,kept,count
0,1,"WB_CCKP_HD30, WB_CCKP_TR29",WB_CCKP_TR29,2
1,2,WB_CCKP_TR32,WB_CCKP_TR32,1
2,3,"WB_CCKP_FD, WB_CCKP_HDD65, WB_CCKP_ID, WB_CCKP_TAS, WB_CCKP_TASMAX, WB_CCKP_TASMIN, WB_CCKP_TNN",WB_CCKP_TNN,7
3,4,"WB_CCKP_CDD65, WB_CCKP_SD, WB_CCKP_TR, WB_CCKP_TR23, WB_CCKP_TR26",WB_CCKP_TR,5
4,5,WB_CCKP_HURS,WB_CCKP_HURS,1
5,6,WB_CCKP_TXX,WB_CCKP_TXX,1
6,7,WB_CCKP_CDD,WB_CCKP_CDD,1
7,8,"WB_CCKP_PR, WB_CCKP_R20MM, WB_CCKP_R95PTOT",WB_CCKP_PR,3
8,9,"WB_CCKP_RX1DAY, WB_CCKP_RX5DAY",WB_CCKP_RX5DAY,2
9,10,WB_CCKP_R50MM,WB_CCKP_R50MM,1


,feature,cluster,abs_corr_with_yield,kept,decision
23,WB_CCKP_TR29,1,0.109367,True,keep: strongest absolute target correlation
5,WB_CCKP_HD30,1,0.097537,False,drop: not selected representative from average-linkage cluster
24,WB_CCKP_TR32,2,0.070299,True,keep: strongest absolute target correlation
19,WB_CCKP_TNN,3,0.322165,True,keep: strongest absolute target correlation
4,WB_CCKP_FD,3,0.280325,False,drop: not selected representative from average-linkage cluster
6,WB_CCKP_HDD65,3,0.150670,False,drop: not selected representative from average-linkage cluster
8,WB_CCKP_ID,3,0.280273,False,drop: not selected representative from average-linkage cluster
16,WB_CCKP_TAS,3,0.108576,False,drop: not selected representative from average-linkage cluster
17,WB_CCKP_TASMAX,3,0.108576,False,drop: not selected representative from average-linkage cluster
18,WB_CCKP_TASMIN,3,0.108576,False,drop: not selected representative from average-linkage cluster


,feature,abs_corr_with_yield
0,WB_CCKP_CDD,0.160530
1,WB_CCKP_CSDI,0.070736
2,WB_CCKP_CWD,0.301749
3,WB_CCKP_HURS,0.273768
4,WB_CCKP_PR,0.246712
5,WB_CCKP_R50MM,0.050002
6,WB_CCKP_RX5DAY,0.012430
7,WB_CCKP_TNN,0.322165
8,WB_CCKP_TR,0.210216
9,WB_CCKP_TR29,0.109367


,selected_climate,with_year
0,14,15


## Variance inflation factors

VIF is computed on the selected numeric climate variables plus YEAR using training rows. Values above 5 and 10 are flagged. The features with VIF between 5 and 10 are retained: none exceeds 10, they are selected representatives from distinct clusters at the specified 0.85 correlation scale, the winning model is tree-based, and the linear baseline is Ridge-regularized.

In [4]:
display(result["vif"])
print("Final numeric feature list:")
print(result["manifest"]["selected_numeric_features"])

Final numeric feature list:
['WB_CCKP_CDD', 'WB_CCKP_CSDI', 'WB_CCKP_CWD', 'WB_CCKP_HURS', 'WB_CCKP_PR', 'WB_CCKP_R50MM', 'WB_CCKP_RX5DAY', 'WB_CCKP_TNN', 'WB_CCKP_TR', 'WB_CCKP_TR29', 'WB_CCKP_TR32', 'WB_CCKP_TX84RR', 'WB_CCKP_TXX', 'WB_CCKP_WSDI', 'YEAR']


,feature,VIF,flag_over_5,flag_over_10
0,WB_CCKP_TR,7.956401,True,False
1,WB_CCKP_PR,6.681598,True,False
2,WB_CCKP_RX5DAY,6.163213,True,False
3,WB_CCKP_TXX,4.337947,False,False
4,WB_CCKP_HURS,4.028249,False,False
5,WB_CCKP_R50MM,3.799127,False,False
6,WB_CCKP_TR29,3.680404,False,False
7,WB_CCKP_CDD,2.966509,False,False
8,WB_CCKP_CWD,2.767446,False,False
9,WB_CCKP_TR32,2.417512,False,False


The selected features and clustering settings are saved to `phase3_feature_manifest.json`. The observed climate-column count is taken from the CSV prefix, so it remains correct if the source differs from the prompt’s stated count.